# Bloque 4. Abstracción y relaciones entre objetos

## ¿Qué vamos a aprender?

Hasta ahora hemos trabajado principalmente con objetos aislados.

En programas reales, los objetos se relacionan entre sí. En este bloque veremos:

- ocultación de información;
- agregación;
- asociación;
- listas que contienen objetos;
- referencias entre objetos.

## Contenido del bloque

| Subbloque | Contenido |
|---|---|
| **4.1** | Abstracción y ocultación de información |
| **4.2** | Agregación |
| **4.3** | Asociación |

# 4.1. Abstracción y ocultación de información

Una clase debería ofrecer al exterior las operaciones necesarias sin obligarnos a conocer todos sus detalles internos.

Por ejemplo, para utilizar una cuenta bancaria queremos poder hacer:

```python
cuenta.ingresar(100)
```

sin preocuparnos de cómo se representa internamente el saldo.

Esta idea está relacionada con la **abstracción** y la **encapsulación**.

## Atributos con doble guion bajo

En el material inicial se utilizaban atributos cuyo nombre comenzaba por `__`.

Ejemplo:

```python
self.__atributo_privado
```

Es importante matizar algo:

En Python esto no convierte el atributo en «privado» de forma absoluta. Python aplica un mecanismo denominado **name mangling**, que dificulta el acceso accidental desde fuera de la clase.

Didácticamente podemos entenderlo como:

> «Este atributo forma parte de la implementación interna de la clase y no deberíamos manipularlo directamente desde fuera».

In [ ]:
class Ejemplo:

    def __init__(self):
        # El doble guion bajo indica que queremos tratar
        # este atributo como parte interna de la clase.
        self.__atributo_privado = (
            "Soy un atributo gestionado internamente por el objeto."
        )

    def get_atributo_privado(self):
        # Método de lectura: devuelve el valor.
        return self.__atributo_privado

    def set_atributo_privado(self, valor):
        # Método de escritura: modifica el valor.
        self.__atributo_privado = valor


e = Ejemplo()

print(e.get_atributo_privado())

e.set_atributo_privado(
    "Ahora he sido modificado mediante un método."
)

print(e.get_atributo_privado())

# 4.2. Agregación

La **agregación** representa una relación «todo-parte».

Un objeto contiene referencias a otros objetos, pero esos objetos pueden seguir teniendo sentido de forma independiente.

Ejemplo:

```text
Catálogo
   ├── Película 1
   ├── Película 2
   └── Película 3
```

El catálogo no guarda simples textos: guarda **objetos `Pelicula`** dentro de una lista.

In [ ]:
class Pelicula:

    def __init__(self, titulo, duracion, lanzamiento):
        self.titulo = titulo
        self.duracion = duracion
        self.lanzamiento = lanzamiento

    def __str__(self):
        return (
            f"{self.titulo} "
            f"({self.lanzamiento}) - "
            f"{self.duracion} min"
        )


class Catalogo:

    def __init__(self, peliculas=None):
        # IMPORTANTE:
        #
        # Evitamos:
        #
        #     def __init__(self, peliculas=[]):
        #
        # porque una lista utilizada como valor por defecto se crea
        # una sola vez y podría compartirse entre instancias.
        if peliculas is None:
            self.peliculas = []
        else:
            # Creamos una copia de la lista recibida.
            self.peliculas = list(peliculas)

    def agregar(self, pelicula):
        # pelicula será un objeto de la clase Pelicula.
        self.peliculas.append(pelicula)

    def mostrar(self):
        if not self.peliculas:
            print("El catálogo está vacío.")
            return

        for pelicula in self.peliculas:
            # print() utilizará automáticamente pelicula.__str__()
            print(pelicula)


# ------------------------------------------------------------
# CREAMOS OBJETOS Pelicula
# ------------------------------------------------------------

p1 = Pelicula(
    titulo="El Padrino",
    duracion=175,
    lanzamiento=1972
)

p2 = Pelicula(
    titulo="El Padrino: Parte II",
    duracion=202,
    lanzamiento=1974
)


# ------------------------------------------------------------
# CREAMOS UN OBJETO Catalogo
# ------------------------------------------------------------

catalogo = Catalogo([p1])

# Agregamos posteriormente otra película.
catalogo.agregar(p2)

catalogo.mostrar()

## ¿Dónde aparece la estructura de datos?

Aquí:

```python
self.peliculas = []
```

Pero ahora la lista no almacena números o cadenas.

Almacena **referencias a objetos `Pelicula`**.

Eso nos permite recorrer la colección y acceder a los atributos de cada objeto.

In [ ]:
print("Títulos almacenados en el catálogo:")

for pelicula in catalogo.peliculas:
    print("-", pelicula.titulo)

# 4.3. Asociación

La **asociación** representa una relación entre objetos.

En nuestro ejemplo:

- un cliente puede tener varias cuentas;
- una cuenta puede tener varios clientes o titulares.

Podemos visualizarlo así:

```text
Cliente  <-------->  Cuenta
```

Cada objeto conserva referencias a objetos de la otra clase.

In [ ]:
class Cliente:

    def __init__(
        self,
        nombre,
        apellidos,
        dni,
        direccion,
        telefono,
        email
    ):
        self.nombre = nombre
        self.apellidos = apellidos
        self.dni = dni
        self.direccion = direccion
        self.telefono = telefono
        self.email = email

        # Lista de objetos Cuenta asociados al cliente.
        self.cuentas = []

    def __str__(self):
        return f"{self.nombre} {self.apellidos} ({self.dni})"

    def agregar_cuenta(self, cuenta):
        # Evitamos añadir dos veces la misma cuenta.
        if cuenta not in self.cuentas:
            self.cuentas.append(cuenta)


class Cuenta:

    def __init__(self, numero, saldo=0):
        self.numero = numero
        self.saldo = saldo

        # Lista de objetos Cliente que son titulares.
        self.clientes = []

    def ingresar(self, cantidad):
        # Solo aceptamos ingresos positivos.
        if cantidad <= 0:
            print("La cantidad debe ser positiva.")
            return

        self.saldo += cantidad

    def agregar_cliente(self, cliente):
        # --------------------------------------------------------
        # MANTENEMOS LA RELACIÓN EN LOS DOS SENTIDOS
        # --------------------------------------------------------

        # La cuenta referencia al cliente.
        if cliente not in self.clientes:
            self.clientes.append(cliente)

        # El cliente referencia a la cuenta.
        if self not in cliente.cuentas:
            cliente.agregar_cuenta(self)

    def __str__(self):
        # Creamos una cadena con los nombres de todos los titulares.
        nombres_titulares = ", ".join(
            f"{cliente.nombre} {cliente.apellidos}"
            for cliente in self.clientes
        )

        if not nombres_titulares:
            nombres_titulares = "Sin titulares"

        return (
            f"Cuenta {self.numero} | "
            f"Titulares: {nombres_titulares} | "
            f"Saldo: {self.saldo:.2f} €"
        )

In [ ]:
# ------------------------------------------------------------
# CREAMOS CLIENTES
# ------------------------------------------------------------

cliente_1 = Cliente(
    nombre="Hector",
    apellidos="Costa",
    dni="11111111A",
    direccion="Calle 1",
    telefono="111111111",
    email="hector.costa@example.com"
)

cliente_2 = Cliente(
    nombre="Ana",
    apellidos="López",
    dni="22222222B",
    direccion="Calle 2",
    telefono="222222222",
    email="ana.lopez@example.com"
)


# ------------------------------------------------------------
# CREAMOS UNA CUENTA
# ------------------------------------------------------------

cuenta_1 = Cuenta(
    numero="ES00123",
    saldo=100
)


# ------------------------------------------------------------
# ESTABLECEMOS LA ASOCIACIÓN
# ------------------------------------------------------------

cuenta_1.agregar_cliente(cliente_1)
cuenta_1.agregar_cliente(cliente_2)

print(cuenta_1)

## Visualizar los titulares de la cuenta


Ahora es fácil recorrer la lista porque `cuenta_1.clientes` contiene objetos `Cliente`.

In [ ]:
print("Titulares de la cuenta:")

for cliente in cuenta_1.clientes:
    print(f"- {cliente.nombre} {cliente.apellidos}")

## Recorrer la relación en el sentido contrario

También podemos consultar las cuentas asociadas a un cliente.

In [ ]:
print(f"Cuentas de {cliente_1.nombre}:")

for cuenta in cliente_1.cuentas:
    print(f"- {cuenta.numero}")

## Resumen del bloque

### Ocultación de información

La clase controla cómo se consulta o modifica su estado interno.

### Agregación

```text
Catálogo
    └── contiene una lista de objetos Pelicula
```

### Asociación

```text
Cliente  <-------->  Cuenta
```

Los objetos mantienen referencias entre sí.

### Conexión con estructuras de datos

Las estructuras de datos pueden contener objetos completos:

```python
self.peliculas = [pelicula_1, pelicula_2]
self.clientes = [cliente_1, cliente_2]
```

Este paso es muy importante porque conecta directamente **POO** con **estructuras de datos**.

---

# Ejercicio del bloque 4. Un curso que contiene alumnos

## Enunciado

Vamos a representar la relación entre un `Curso` y sus `Alumno`.

Crea dos clases:

### Clase `Alumno`

Debe tener:

- `nombre`
- `email`

y un método `__str__()` que permita mostrar al alumno de forma legible.

### Clase `Curso`

Debe tener:

- `nombre`
- una lista de alumnos inicialmente vacía.

Debe incluir los métodos:

1. `agregar_alumno(alumno)`  
   Añade un objeto `Alumno` a la lista.

2. `mostrar_alumnos()`  
   Recorre la lista y muestra todos los alumnos matriculados.

Después:

3. Crea tres objetos `Alumno`.
4. Crea un objeto `Curso` llamado `"Programación en Python"`.
5. Añade los tres alumnos al curso.
6. Muestra todos los alumnos matriculados.

## Condición importante

La lista de alumnos debe crearse dentro de `__init__`.

No utilices:

```python
def __init__(self, alumnos=[]):
```

> **Objetivo del ejercicio:** practicar una relación entre objetos y utilizar una lista como estructura de datos para almacenar objetos.

In [ ]:
# Escribe aquí tu solución